In [4]:
import os
import pandas as pd
import re
import yaml  # pip install pyyaml

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\July_ 21\Config Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "3.1_YML_List_ShallowC.csv")

# === UNIVERSAL UNIT TEST KEYWORDS ===
UNIT_TEST_KEYWORDS = [
    'gradlew test', './gradlew test', 'testdebugunittest', 'testreleaseunittest',
    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

# === CI-SPECIFIC INSTRUMENTATION RULES ===
def detect_instrumentation_by_platform(ci_platform, yaml_text, run_scripts, parsed):
    """
    Improved detection:
    - Wrappers & GMD: parsed 'uses'
    - Manual emulator: must have (sdkmanager or avdmanager) AND (emulator or avdmanager)
    - No more adb-only false positives.
    """
    run_text = "\n".join(run_scripts)
    found = set()
    matched = []

    # === Common emulator keywords ===
    has_sdk = any(kw in run_text for kw in ['sdkmanager'])
    has_avd = any(kw in run_text for kw in ['avdmanager'])
    has_emulator = any(kw in run_text for kw in ['emulator'])
    has_create_avd = any(kw in run_text for kw in ['android create avd'])

    if ci_platform == "GitHub_Actions":
        # 1️⃣ Wrappers & GMD
        for job in parsed.get('jobs', {}).values():
            for step in job.get('steps', []):
                if isinstance(step, dict) and 'uses' in step:
                    uses = step['uses'].lower()
                    if 'reactivecircus/android-emulator-runner' in uses:
                        found.add('GitHub_emulator_full')
                        matched.append('reactivecircus/android-emulator-runner')
                    if 'malinskiy/action-android/emulator-run-cmd' in uses:
                        found.add('GitHub_emulator_compact')
                        matched.append('malinskiy/action-android/emulator-run-cmd')
                    if any(gmd in uses for gmd in ['cleanmanageddevices', 'managedvirtualdevice', 'manageddevices']):
                        found.add('GitHub_GMD')
                        matched.append('GMD keywords (uses)')

        if any("cleanmanageddevices" in s or "manageddevices" in s for s in run_scripts):
            found.add('GitHub_GMD')
            matched.append('GMD keywords (Gradle run)')

        # ✅ Improved manual check:
        if (has_sdk or has_avd) and (has_emulator or has_avd):
            found.add('GitHub_emulator_manual')
            matched.append('manual emulator setup')

    elif ci_platform == "Circle_CI":
        if (has_sdk or has_avd) and (has_emulator or has_avd):
            found.add('Circle_CI_emulator_manual')
            matched.append('manual emulator setup')

    elif ci_platform == "Bitrise":
        if (has_sdk or has_avd) and (has_emulator or has_avd):
            found.add('Bitrise_emulator_manual')
            matched.append('manual emulator setup')

    elif ci_platform == "GitLab":
        if (has_sdk or has_avd) and (has_emulator or has_avd):
            found.add('GitLab_emulator_manual')
            matched.append('manual emulator setup')

    elif ci_platform == "Travis_CI":
        if (has_sdk or has_avd or has_create_avd) and (has_emulator or has_avd):
            found.add('Travis_CI_emulator_manual')
            matched.append('manual emulator setup')

    elif ci_platform == "Other":
        if (has_sdk or has_avd) and (has_emulator or has_avd):
            found.add('Other_emulator_manual')
            matched.append('manual emulator setup')

    # === Third-party Labs ===
    lab_run_text = run_text
    for job in parsed.get('jobs', {}).values():
        for step in job.get('steps', []):
            if isinstance(step, dict) and 'uses' in step:
                uses = step['uses'].lower()
                if 'firebase-test-lab-action' in uses:
                    found.add('Firebase_Compact')
                    matched.append('Firebase-Test-Lab-Action')
                if 'microsoft/appcenter-test-cli-action' in uses:
                    found.add('Appcenter')
                    matched.append('Appcenter Test CLI Action')
                if 'browserstack' in uses or 'browserstack/github-actions' in uses:
                    found.add('Browserstack')
                    matched.append('Browserstack action')
                if 'saucelabs/saucectl-run-action' in uses:
                    found.add('SauceLabs')
                    matched.append('Sauce Labs GitHub Action')

    if 'gcloud firebase test android run' in lab_run_text:
        found.add('Firebase_Full')
        matched.append('gcloud firebase test android run')

    if 'appcenter test run' in lab_run_text:
        found.add('Appcenter')
        matched.append('appcenter test run')

    if 'browserstack' in lab_run_text:
        found.add('Browserstack')
        matched.append('browserstack CLI call')

    if 'saucectl' in lab_run_text:
        found.add('SauceLabs')
        matched.append('saucectl CLI')

    return found, matched


def detect_ci_platform(file_path, yaml_text):
    path = file_path.lower()
    text = yaml_text.lower()
    
    # Match against CI-specific patterns
    ci_patterns = {
    r'\.travis\.yml$': 'Travis_CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'Circle_CI',
    r'\.circleci/config\.yml$': 'Circle_CI',
    r'azure-pipelines\.yml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}

    for pattern, ci_name in ci_patterns.items():
        if re.search(pattern, path):
            return ci_name

    # Fallbacks if pattern not matched but keywords exist
    if 'uses: actions/' in text:
        return "GitHub_Actions"
    if 'uses: docker://circleci/' in text or 'circleci/android:' in text:
        return "Circle_CI"
    if 'travis' in text:
        return "Travis_CI"
    if 'bitrise' in text:
        return "Bitrise"
    if 'gitlab-ci' in text:
        return "GitLab"
    if 'jenkins' in text:
        return "Jenkins"
    if 'semaphore' in text:
        return "Semaphore"
    if 'saucelabs' in text:
        return "SauceLabs"
    if 'appcenter' in text:
        return "Appcenter"
    if 'browserstack' in text:
        return "Browserstack"

    return "Other"


def detect_unit_test(yaml_text):
    text = yaml_text.lower()
    for kw in UNIT_TEST_KEYWORDS:
        if kw in text:
            return True
    return False


def extract_run_scripts(parsed):
    def get_runs(node):
        runs = []
        if isinstance(node, dict):
            for k, v in node.items():
                if k == 'run' and isinstance(v, str):
                    runs.append(v.lower())
                else:
                    runs.extend(get_runs(v))
        elif isinstance(node, list):
            for item in node:
                runs.extend(get_runs(item))
        return runs

    scripts = get_runs(parsed)
    if isinstance(parsed, dict):
        for key in ['script', 'before_script', 'before_install', 'after_script']:
            val = parsed.get(key)
            if isinstance(val, list):
                scripts.extend([str(v).lower() for v in val])
            elif isinstance(val, str):
                scripts.append(val.lower())
    return scripts


# === PROCESS ALL ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)

            # ✅ NEW: Extract full_name between first and third '__'
            parts = filename.split("__")
            if len(parts) >= 3:
                full_name = f"{parts[1]}__{parts[2]}"
            else:
                full_name = filename  # fallback

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)

                    parsed = yaml.safe_load(raw)
                    run_scripts = extract_run_scripts(parsed)

                    is_unit = detect_unit_test(raw)
                    instr_types, matched_keys = detect_instrumentation_by_platform(ci_platform, raw, run_scripts, parsed)

                    test_type_str = ', '.join(sorted(instr_types))
                    is_instr = bool(instr_types)

                    results.append({
                        'full_name': full_name,
                        'ci_platform': ci_platform,
                        'test_type': test_type_str,
                        'unit_test': is_unit,
                        'instrumentation_test': is_instr,
                        'matched_keywords': "; ".join(sorted(set(matched_keys)))
                    })

                    print(f"✅ {filename} | CI: {ci_platform} | Unit: {is_unit} | Instr: {is_instr} | Keys: {matched_keys}")

            except Exception as e:
                print(f"❌ ERROR: {file_path} => {e}")
                results.append({
                    'filename': filename,
                    'full_name': full_name,
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False,
                    'matched_keywords': ''
                })

# === EXPORT ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ DONE! CSV generated at: {OUTPUT_CSV}")


ModuleNotFoundError: No module named 'pandas'